In [1]:
import threading
import time
from pathlib import Path

import torch
import numpy as np
import tqdm

from diff_mfld.geodesic.geodesic_funcs import ExpMethod, LogMethod
from diff_mfld.geometry.metric import MetricField
from diff_mfld.mfld import ComputeMfld
from optim.subsolver_methods import SubsolverMethod
from optim.subsolvers.rgd import RiemGradDescentCfg
from optim.subsolvers.rtr import RiemTrustRegionCfg
from problem import create_problem
from testing.testing_metrics import euclid_metric, scaled_metric, coupled_metric, asymmetric_metric

In [2]:
methods = [
    ((ExpMethod.IVP, LogMethod.BVP), "ivpbvp"),
    # ((ExpMethod.APPROX_O1, LogMethod.APPROX_O1), "o1"),
    # ((ExpMethod.APPROX_O2, LogMethod.APPROX_O2), "o2"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O3), "o3"),
    # ((ExpMethod.APPROX_O2, LogMethod.APPROX_O1), "o2_o1"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O1), "o3_o1"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O2), "o3_o2"),
]

# linear scaling of the domain
max_domain_scaling = [
    0.5, 1., 1.5, 1.75
]

# parameters for random starting positions
p0_mean = np.array([2., -5., -8.])
p0_covar = 2. ** 2 * np.eye(3)
num_p0 = 20

# cost minimized at this point
cost_center = np.array([-5., -1., 3.])

# metrics to use in testing

metric_funcs = [
    (euclid_metric, "euclid"),
    (scaled_metric, "scaled"),
    (coupled_metric, "coupled"),
    (asymmetric_metric, "asymmetric"),
]


In [3]:
# generates the random points and scales them

# "To Everything? To the great Question of Life, the Universe and everything?"
r = np.random.default_rng(42)
p_rand_starting = r.multivariate_normal(p0_mean, p0_covar, num_p0)
max_starting_norm = np.max(np.linalg.vector_norm(p_rand_starting, axis=1))

# scales the optimization functions and starting points
p_rand_starting /= max_starting_norm
cost_center /= max_starting_norm

In [4]:
# configure the (unconstrained) configurations
optimizers = [
    ((SubsolverMethod.RIEM_GRAD_DESCENT, RiemGradDescentCfg()), "rgd"),
    ((SubsolverMethod.RIEM_TRUST_REGION, RiemTrustRegionCfg()), "rtr")
]

In [5]:
# root directory to store output files inside
base_data_dir = Path("../data/unconstrained")

In [6]:
from diff_mfld.mfld import Mfld
import itertools

# run the optimization scheme

cfgs = itertools.product(
    metric_funcs, max_domain_scaling, optimizers, methods
)
cfg_pbar = tqdm.tqdm(cfgs, desc="Configurations")

for (
        (metric_fn, metric_fn_label),
        scaling,
        ((optimizer, optimizer_cfg), optimizer_label),
        ((exp_method, log_method), method_label)
) in cfg_pbar:

    # setup the manifold configuration
    metric = MetricField(lambda x1, x2, x3: metric_fn(x1, x2, x3, scaling=scaling))
    conn = metric.christoffels()  # Levi-Civita connection (metric-compatible and torsion-free)
    mfld = Mfld(metric, conn)

    compute_mfld = ComputeMfld(
        mfld=mfld,
        exp_method=exp_method,
        log_method=log_method,
        dist_method=log_method,  # which logarithm to use when computing distance
    )

    # scale the problem coordinates accordingly
    trials_p_starting = p_rand_starting * scaling
    trials_cost_center = cost_center * scaling

    # print(f"trials_cost_center: {trials_cost_center}")
    # assert False

    # create the problem
    cost, _ = create_problem(torch.tensor(trials_cost_center))

    trials_pbar = tqdm.tqdm(range(trials_p_starting.shape[0]), desc="Trials")
    for start_idx in trials_pbar:
        p0 = torch.tensor(trials_p_starting[start_idx])
        result = optimizer(cost, p0, compute_mfld, optimizer_cfg)
        print(result)

        # save the resulting dataclass
        filename = f"{optimizer_label}/metric_{metric_fn_label}__scaling_{scaling}__trial_{start_idx}__geod_method_{method_label}.pt"
        file_path = base_data_dir / filename
        print(f"\tSaving to {file_path}")

        torch.save(result, file_path)

Configurations: 0it [00:00, ?it/s]
Trials:   5%|▌         | 1/20 [00:01<00:30,  1.60s/it]

RiemGradDescentResult(success=True, p=tensor([-0.1521, -0.0522,  0.0771]), iters=7, history=RiemGradDescentHistory(p_hist=tensor([[ 0.0114, -0.1828, -0.1269],
        [-0.0442, -0.1384, -0.0575],
        [-0.0831, -0.1073, -0.0090],
        [-0.1103, -0.0855,  0.0250],
        [-0.1294, -0.0703,  0.0488],
        [-0.1427, -0.0596,  0.0655],
        [-0.1521, -0.0522,  0.0771]]), f_hist=tensor([0.0137, 0.0067, 0.0033, 0.0016, 0.0008, 0.0004, 0.0002])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_0.5__trial_0__geod_method_ivpbvp.pt



Trials:  10%|█         | 2/20 [00:03<00:30,  1.72s/it]

RiemGradDescentResult(success=True, p=tensor([-0.1561, -0.0506,  0.0770]), iters=8, history=RiemGradDescentHistory(p_hist=tensor([[ 0.0423, -0.2271, -0.2268],
        [-0.0225, -0.1694, -0.1275],
        [-0.0679, -0.1290, -0.0579],
        [-0.0997, -0.1007, -0.0093],
        [-0.1220, -0.0810,  0.0248],
        [-0.1375, -0.0671,  0.0487],
        [-0.1484, -0.0574,  0.0654],
        [-0.1561, -0.0506,  0.0770]]), f_hist=tensor([0.0242, 0.0118, 0.0058, 0.0028, 0.0014, 0.0007, 0.0003, 0.0002])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_0.5__trial_1__geod_method_ivpbvp.pt



Trials:  15%|█▌        | 3/20 [00:05<00:28,  1.68s/it]

...

RiemTrustRegionResult(success=True, p=tensor([ 0.3971, -0.9632, -1.0514]), iters=2, history=RiemTrustRegionHistory(p_hist=tensor([[ 0.3971, -0.9632, -1.0514],
        [ 0.3971, -0.9632, -1.0514]]), f_hist=tensor([2.3840, 2.3840]), quality_hist=tensor([-1.1457, -1.4529]), radius_hist=tensor([0.1250, 0.0312])))
	Saving to ../data/unconstrained/rtr/metric_asymmetric__scaling_1.75__trial_17__geod_method_ivpbvp.pt



Trials:  95%|█████████▌| 19/20 [01:22<00:04,  4.31s/it]

RiemTrustRegionResult(success=True, p=tensor([-0.6081, -0.1217,  0.3651]), iters=7, history=RiemTrustRegionHistory(p_hist=tensor([[-0.0084, -0.6460, -0.7994],
        [-0.1355, -0.5353, -0.5703],
        [-0.2651, -0.4274, -0.3386],
        [-0.3918, -0.3216, -0.0994],
        [-0.5110, -0.2155,  0.1467],
        [-0.6081, -0.1217,  0.3651],
        [-0.6081, -0.1217,  0.3651]]), f_hist=tensor([3.2815e+00, 2.0318e+00, 1.0791e+00, 4.4608e-01, 9.4546e-02, 1.2631e-07,
        1.2631e-07]), quality_hist=tensor([0.7089, 1.0811, 1.0791, 1.0385, 1.0123, 1.0000, 0.0326]), radius_hist=tensor([0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.1250])))
	Saving to ../data/unconstrained/rtr/metric_asymmetric__scaling_1.75__trial_18__geod_method_ivpbvp.pt



Trials: 100%|██████████| 20/20 [01:24<00:00,  4.25s/it]
Configurations: 32it [41:46, 78.34s/it] 

RiemTrustRegionResult(success=True, p=tensor([ 0.6073, -0.8193, -0.7379]), iters=2, history=RiemTrustRegionHistory(p_hist=tensor([[ 0.6073, -0.8193, -0.7379],
        [ 0.6073, -0.8193, -0.7379]]), f_hist=tensor([1.3560, 1.3560]), quality_hist=tensor([-1.3632, -1.5888]), radius_hist=tensor([0.1250, 0.0312])))
	Saving to ../data/unconstrained/rtr/metric_asymmetric__scaling_1.75__trial_19__geod_method_ivpbvp.pt
